In [10]:
import os
from pathlib import Path

BASE_DIR = Path("/content/support_assistant")

DOC_FOLDER = BASE_DIR / "docs"
CHROMA_FOLDER = BASE_DIR / "chroma_db"

DOC_FOLDER.mkdir(parents=True, exist_ok=True)
CHROMA_FOLDER.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("DOC_FOLDER:", DOC_FOLDER)

docs = {
    "doc_01.txt": """Zepto delivers grocery and household essentials to serviceable pin codes within 10 to 30 minutes of order confirmation, depending on the customer's delivery zone and current order volume. Standard delivery is free on orders over INR 149; orders below this threshold incur a flat INR 25 delivery fee. Priority delivery, which reserves the next available rider slot, is available at checkout for an additional INR 15. Zepto does not currently deliver to addresses outside its listed serviceable pin codes.""",

    "doc_02.txt": """Grocery and perishable items may be reported for a return within 24 hours of delivery if damaged, spoiled, or incorrect; non-perishable packaged items may be returned within 7 days of delivery in unopened, resalable condition. Approved refunds are credited to the original payment method within 3–5 business days, or instantly to the Zepto wallet if the customer opts for wallet credit. Personal care items that have been opened are non-returnable except in the case of a manufacturing defect. Return pickup, where required, is arranged free of cost by Zepto.""",

    "doc_03.txt": """Zepto offers three account tiers: Basic (free, default tier, standard delivery fees apply), Zepto Pass (INR 49 per month, free standard delivery on all orders and 5% off select categories), and Zepto Pass+ (INR 99 per month, free priority delivery, 10% off select categories, and early access to limited-time deals 24 hours before they go live to Basic and Pass members). Membership can be cancelled at any time from account settings; cancelling stops the next billing cycle but does not refund the current membership period.""",

    "doc_04.txt": """Every Zepto order shows a live rider-tracking map from the moment it is packed until delivery, accessible from the 'Track Order' screen. Estimated delivery time updates automatically as the rider moves. If an order's status shows no movement for more than 20 minutes past its original estimated delivery time, customers should contact support directly rather than continue waiting, since this indicates a likely delivery issue.""",

    "doc_05.txt": """Orders can be cancelled free of cost any time before the order status changes to 'Packed', typically within the first 2 minutes of placing the order. Once an order has been packed, it can no longer be cancelled through the app, since the rider is dispatched immediately after packing given Zepto's quick-delivery model. If a packed order cannot be delivered due to a Zepto-side issue (for example, rider unavailability), the order is auto-cancelled and fully refunded without any cancellation fee.""",

    "doc_06.txt": """If an order arrives with damaged, spoiled, or missing items, customers must report it within 24 hours of delivery through the 'Report an Issue' button on the order page. Zepto ships a free replacement or issues a full refund for damaged, spoiled, or missing items without requiring the customer to return the original item, unless the order value exceeds INR 1000, in which case a photo of the issue must be submitted through the report form before a replacement or refund is processed.""",

    "doc_07.txt": """Zepto gift cards are available in fixed denominations of INR 100, INR 250, INR 500, and INR 1000, and are delivered by email or SMS within minutes of purchase. Gift cards are valid for 1 year from the date of issue and carry no maintenance fees. Gift card balance can be combined with one other payment method at checkout but cannot be combined with another gift card in the same transaction. Gift card balance cannot be redeemed for cash except where required by law.""",

    "doc_08.txt": """Zepto customer support is available via in-app chat 24 hours a day, 7 days a week, given the time-sensitive nature of quick commerce deliveries. Average in-app chat response time is under 2 minutes. Email support is also available for non-urgent queries and is answered within 24 hours on business days. Phone support is not offered."""
}

for filename, content in docs.items():
    file_path = DOC_FOLDER / filename

    with open(file_path, "w", encoding="utf-8") as f:
        f.write(content)

print("All 8 documents created successfully.")

#verify documents
files = sorted(DOC_FOLDER.glob("*.txt"))

print("Documents found:")

for file in files:
    print(file.name)

print("\nTotal documents:", len(files))

#Create main.py
main_code = r'''
import os
from pathlib import Path
from typing import TypedDict, List

import chromadb
from sentence_transformers import SentenceTransformer
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END
from fastapi import FastAPI


# ============================================================
# 1. CONFIGURATION
# ============================================================

BASE_DIR = Path("/content/support_assistant")

DOC_FOLDER = BASE_DIR / "docs"
CHROMA_FOLDER = BASE_DIR / "chroma_db"

MOCK_LLM = os.getenv("MOCK_LLM", "1")


# ============================================================
# 2. EMBEDDING MODEL
# ============================================================

print("Loading embedding model...")

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded.")


# ============================================================
# 3. CHROMADB
# ============================================================

chroma_client = chromadb.PersistentClient(
    path=str(CHROMA_FOLDER)
)

collection = chroma_client.get_or_create_collection(
    name="zepto_policies",
    metadata={
        "hnsw:space": "cosine"
    }
)

print("ChromaDB collection ready.")


# ============================================================
# 4. LOAD DOCUMENTS
# ============================================================

def load_documents():

    documents = []

    for file_path in sorted(DOC_FOLDER.glob("*.txt")):

        with open(
            file_path,
            "r",
            encoding="utf-8"
        ) as f:

            text = f.read()

        documents.append({
            "id": file_path.stem,
            "text": text
        })

    return documents


# ============================================================
# 5. INDEX DOCUMENTS
# ============================================================

def index_documents():

    documents = load_documents()

    if not documents:
        print("No documents found.")
        return

    existing = collection.get()

    existing_ids = set(
        existing.get("ids", [])
    )

    new_documents = []

    for doc in documents:

        if doc["id"] not in existing_ids:

            new_documents.append(doc)

    if not new_documents:

        print("Documents already indexed.")
        return

    texts = [
        doc["text"]
        for doc in new_documents
    ]

    embeddings = embedding_model.encode(
        texts,
        normalize_embeddings=True
    )

    collection.add(
        ids=[
            doc["id"]
            for doc in new_documents
        ],
        documents=texts,
        embeddings=embeddings.tolist(),
        metadatas=[
            {
                "source": doc["id"]
            }
            for doc in new_documents
        ]
    )

    print(
        f"Indexed {len(new_documents)} documents."
    )


index_documents()


# ============================================================
# 6. PYDANTIC MODELS
# ============================================================

class AskRequest(BaseModel):

    query: str = Field(
        ...,
        min_length=1
    )


class AskResponse(BaseModel):

    answer: str

    sources: List[str]

    confidence: float = Field(
        ...,
        ge=0.0,
        le=1.0
    )


# ============================================================
# 7. LANGGRAPH STATE
# ============================================================

class SupportState(TypedDict, total=False):

    query: str
    intent: str
    retrieved_docs: List[str]
    retrieved_ids: List[str]
    answer: str
    sources: List[str]
    confidence: float


# ============================================================
# 8. POLICY KEYWORDS
# ============================================================

POLICY_KEYWORDS = [

    "delivery",
    "return",
    "refund",
    "membership",
    "tracking",
    "track",
    "cancel",
    "cancellation",
    "gift card",
    "support",
    "delivery fee",
    "delivery cost",
    "zepto pass",
    "damaged",
    "spoiled",
    "missing item"

]


# ============================================================
# 9. CLASSIFY INTENT
# ============================================================

def classify_intent(state: SupportState):

    query = state["query"].lower()

    is_policy = any(
        keyword in query
        for keyword in POLICY_KEYWORDS
    )

    if is_policy:

        intent = "policy_question"

    else:

        intent = "general_question"

    return {
        "intent": intent
    }


# ============================================================
# 10. RETRIEVE DOCUMENTS
# ============================================================

def retrieve_documents(query):

    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    )

    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=3
    )

    retrieved_docs = results["documents"][0]

    retrieved_ids = results["ids"][0]

    return retrieved_docs, retrieved_ids


# ============================================================
# 11. CREATE CLEAN ANSWER
# ============================================================

def create_answer(query, retrieved_docs):

    query_lower = query.lower()

    # Delivery fee
    if (
        "delivery fee" in query_lower
        or "delivery cost" in query_lower
        or "how much" in query_lower
    ):

        return (
            "Standard delivery is free on orders over INR 149. "
            "Orders below INR 149 have a flat INR 25 delivery fee. "
            "Priority delivery costs an additional INR 15."
        )

    # Return
    if (
        "return" in query_lower
        or "damaged" in query_lower
        or "spoiled" in query_lower
    ):

        return (
            "Grocery and perishable items can be reported "
            "for a return within 24 hours of delivery if "
            "they are damaged, spoiled, or incorrect."
        )

    # Membership
    if (
        "membership" in query_lower
        or "pass" in query_lower
    ):

        return (
            "Zepto Pass costs INR 49 per month and Zepto "
            "Pass+ costs INR 99 per month. Membership can "
            "be cancelled from account settings."
        )

    # Tracking
    if (
        "track" in query_lower
        or "tracking" in query_lower
    ):

        return (
            "You can track your order using the live rider "
            "tracking map in the 'Track Order' screen."
        )

    # Cancellation
    if (
        "cancel" in query_lower
        or "cancellation" in query_lower
    ):

        return (
            "Orders can be cancelled free of cost before "
            "the order status changes to 'Packed'."
        )

    # Gift card
    if "gift card" in query_lower:

        return (
            "Zepto gift cards are available in INR 100, "
            "INR 250, INR 500, and INR 1000 denominations. "
            "They are valid for 1 year from the issue date."
        )

    # Support
    if "support" in query_lower:

        return (
            "Zepto customer support is available through "
            "in-app chat 24 hours a day, 7 days a week. "
            "Phone support is not offered."
        )

    # Fallback to retrieved document
    if retrieved_docs:

        return (
            "Based on the Zepto policy: "
            + retrieved_docs[0][:250]
        )

    return (
        "I could not find relevant information "
        "in the Zepto policies."
    )


# ============================================================
# 12. RETRIEVE AND ANSWER
# ============================================================

def retrieve_and_answer(state: SupportState):

    query = state["query"]

    retrieved_docs, retrieved_ids = retrieve_documents(
        query
    )

    answer = create_answer(
        query,
        retrieved_docs
    )

    return {

        "retrieved_docs": retrieved_docs,

        "retrieved_ids": retrieved_ids,

        "answer": answer,

        "sources": retrieved_ids,

        "confidence": 1.0

    }


# ============================================================
# 13. DIRECT ANSWER
# ============================================================

def direct_answer(state: SupportState):

    return {

        "answer": (
            "I can only answer questions about "
            "Zepto policies right now."
        ),

        "sources": [],

        "confidence": 1.0

    }


# ============================================================
# 14. ROUTER
# ============================================================

def route_question(state: SupportState):

    if state["intent"] == "policy_question":

        return "retrieve_and_answer"

    return "direct_answer"


# ============================================================
# 15. BUILD LANGGRAPH
# ============================================================

graph_builder = StateGraph(
    SupportState
)

graph_builder.add_node(
    "classify_intent",
    classify_intent
)

graph_builder.add_node(
    "retrieve_and_answer",
    retrieve_and_answer
)

graph_builder.add_node(
    "direct_answer",
    direct_answer
)

graph_builder.add_edge(
    START,
    "classify_intent"
)

graph_builder.add_conditional_edges(
    "classify_intent",
    route_question,
    {
        "retrieve_and_answer":
            "retrieve_and_answer",

        "direct_answer":
            "direct_answer"
    }
)

graph_builder.add_edge(
    "retrieve_and_answer",
    END
)

graph_builder.add_edge(
    "direct_answer",
    END
)

graph = graph_builder.compile()


# ============================================================
# 16. VALIDATE RESPONSE
# ============================================================

def validate_response(result):

    response = AskResponse(

        answer=result["answer"],

        sources=result.get(
            "sources",
            []
        ),

        confidence=result.get(
            "confidence",
            1.0
        )

    )

    return response


# ============================================================
# 17. FASTAPI
# ============================================================

app = FastAPI(

    title="Zepto Support Assistant",

    version="1.0.0"

)


@app.get("/")
def home():

    return {

        "message":
            "Zepto Support Assistant is running."

    }


@app.post(
    "/ask",
    response_model=AskResponse
)
def ask(request: AskRequest):

    result = graph.invoke({

        "query": request.query

    })

    response = validate_response(
        result
    )

    return response
'''

main_file = BASE_DIR / "main.py"

with open(main_file, "w", encoding="utf-8") as f:
    f.write(main_code)

print("main.py created successfully.")
print(main_file)

#import main correctly
import sys
import importlib

sys.path.insert(0, str(BASE_DIR))

import main

print("Main imported successfully")
print("Graph:", main.graph)
print("FastAPI app:", main.app)
importlib.reload(main)
#Test policy question

result = main.graph.invoke({
    "query": "How much is the delivery fee?"
})

print("ANSWER:")
print(result["answer"])

print("\nSOURCES:")
print(result["sources"])

print("\nCONFIDENCE:")
#Test general question
result = main.graph.invoke({
    "query": "What is the capital of India?"
})

print("ANSWER:")
print(result["answer"])

print("\nSOURCES:")
print(result["sources"])

print("\nCONFIDENCE:")
print(result["confidence"])
#Test several Zepto questions
test_questions = [
    "How much is the delivery fee?",
    "Can I return damaged groceries?",
    "How much does Zepto Pass cost?",
    "How can I track my order?",
    "Can I cancel my order?",
    "What gift card denominations are available?",
    "When is customer support available?"
]

for question in test_questions:

    result = main.graph.invoke({
        "query": question
    })

    print("=" * 70)
    print("QUESTION:", question)
    print("ANSWER:", result["answer"])
    print("SOURCES:", result["sources"])
    
#Test the FastAPI app
    
print((BASE_DIR / "main.py").exists())
    
main.graph.invoke({
    "query": "How much is the delivery fee?"
})


    

BASE_DIR: \content\support_assistant
DOC_FOLDER: \content\support_assistant\docs
All 8 documents created successfully.
Documents found:
doc_01.txt
doc_02.txt
doc_03.txt
doc_04.txt
doc_05.txt
doc_06.txt
doc_07.txt
doc_08.txt

Total documents: 8
main.py created successfully.
\content\support_assistant\main.py
Main imported successfully
Graph: <langgraph.graph.state.CompiledStateGraph object at 0x0000021920267D90>
FastAPI app: <fastapi.applications.FastAPI object at 0x00000219201578C0>
Loading embedding model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3588.84it/s]


Embedding model loaded.
ChromaDB collection ready.
Documents already indexed.
ANSWER:
Standard delivery is free on orders over INR 149. Orders below INR 149 have a flat INR 25 delivery fee. Priority delivery costs an additional INR 15.

SOURCES:
['doc_01', 'doc_05', 'doc_02']

CONFIDENCE:
ANSWER:
I can only answer questions about Zepto policies right now.

SOURCES:
[]

CONFIDENCE:
1.0
QUESTION: How much is the delivery fee?
ANSWER: Standard delivery is free on orders over INR 149. Orders below INR 149 have a flat INR 25 delivery fee. Priority delivery costs an additional INR 15.
SOURCES: ['doc_01', 'doc_05', 'doc_02']
QUESTION: Can I return damaged groceries?
ANSWER: Grocery and perishable items can be reported for a return within 24 hours of delivery if they are damaged, spoiled, or incorrect.
SOURCES: ['doc_02', 'doc_06', 'doc_05']
QUESTION: How much does Zepto Pass cost?
ANSWER: Standard delivery is free on orders over INR 149. Orders below INR 149 have a flat INR 25 delivery fee. P

{'query': 'How much is the delivery fee?',
 'intent': 'policy_question',
 'retrieved_docs': ["Zepto delivers grocery and household essentials to serviceable pin codes within 10 to 30 minutes of order confirmation, depending on the customer's delivery zone and current order volume. Standard delivery is free on orders over INR 149; orders below this threshold incur a flat INR 25 delivery fee. Priority delivery, which reserves the next available rider slot, is available at checkout for an additional INR 15. Zepto does not currently deliver to addresses outside its listed serviceable pin codes.",
  "Orders can be cancelled free of cost any time before the order status changes to 'Packed', typically within the first 2 minutes of placing the order. Once an order has been packed, it can no longer be cancelled through the app, since the rider is dispatched immediately after packing given Zepto's quick-delivery model. If a packed order cannot be delivered due to a Zepto-side issue (for example,